# Предсказание оценки отеля по отзывам

Метрика MAPE

Hotel_Address         -  Адрес отеля, object<br>

Review_Date           -  Дата публикации отзыва, object<br>

Hotel_Name            -  Название отеля, object<br>

Reviewer_Nationality  -  Национальность рецензента, object<br>

Negative_Review       -  Отрицательный отзыв, оставленный рецензентом отелю. В случае отсутствия заполняется значением "No Negative", object<br>

Review_Total_Negative_Word_Counts - Количество слов в отрицательном отзыве, int64<br>

Positive_Review       -  Положительный отзыв, оставленный рецензентом отелю. В случае отсутствия заполняется значением "No Positive", object<br>

Review_Total_Positive_Word_Counts - Количество слов в положительном отзыве, int64<br>

Total_Number_of_Reviews_Reviewer_Has_Given - Количество отзывов, написанных рецензентом в прошлом, int64<br>

Total_Number_of_Reviews - Количество отзывов об отеле, int64<br>

Tags                  -  Теги, данные рецензентом отелю, object<br>

days_since_review     -  Количество дней между написанием отзыва и чисткой, int64<br>

Additional_number_of_soring - Средний балл отеля, на основе всех оценок - с тектом отзыва и без, int64<br>

lat                   -  Широта отеля, float64<br>

lng                   -  Протяженность отеля, float64<br>

**Reviewer_Score**        -  Оценка, данная рецензентом отелю, float64. Целевая переменная

## Загрузка и ознакомление с данными

In [15]:
import numpy as np, pandas as pd, gc, os, re, math
import sys
sys.modules['clearml'] = None
os.environ['CLEARML_NO_TASK'] = '1'
os.environ["CLEARML_DISABLE_AUTO_LOGGING"] = "1"
os.environ["CLEARML_CONFIG_FILE"] = "nul"
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch
import pickle
import optuna
import lightgbm as lgb
from transformers import pipeline
from datasets import Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error
from optuna.integration import XGBoostPruningCallback

from dotenv import load_dotenv
load_dotenv()

np.random.seed(42)

In [16]:
train = pd.read_csv(os.getenv('TRAIN_CSV'))          
test = pd.read_csv(os.getenv('TEST_CSV'))

In [17]:
train.head(2)

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,Ndsm Plein 28 Amsterdam Noord 1033 WB Amsterda...,170,4/17/2017,DoubleTree by Hilton Hotel Amsterdam NDSM Wharf,United Kingdom,Too far from attractions Had to use ferry to ...,72,1593,Staff were very helpful Good breakfast,8,1,5.4,"[' Leisure trip ', ' Couple ', ' Queen Guest R...",108 day,52.400181,4.893665
1,Ferdinand Bolstraat 194 Oud Zuid 1072 LW Amste...,114,5/26/2016,Savoy Hotel Amsterdam,Malaysia,Staff should handle customer document during ...,246,995,Very clean and cozy room Friendly and helpful...,41,1,9.6,"[' Leisure trip ', ' Couple ', ' Small Double ...",434 day,52.349743,4.891191


In [18]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 412590 entries, 0 to 412589
Data columns (total 16 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               412590 non-null  object 
 1   Additional_Number_of_Scoring                412590 non-null  int64  
 2   Review_Date                                 412590 non-null  object 
 3   Hotel_Name                                  412590 non-null  object 
 4   Reviewer_Nationality                        412590 non-null  object 
 5   Negative_Review                             412590 non-null  object 
 6   Review_Total_Negative_Word_Counts           412590 non-null  int64  
 7   Total_Number_of_Reviews                     412590 non-null  int64  
 8   Positive_Review                             412590 non-null  object 
 9   Review_Total_Positive_Word_Counts           412590 non-null  int64  
 

In [19]:
test.head(2)

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Tags,days_since_review,lat,lng
0,7 Western Gateway Royal Victoria Dock Newham L...,359,8/14/2015,Novotel London Excel,United Kingdom,No Negative,0,1158,Excellent location for Excel centre Friendly ...,14,5,"[' Leisure trip ', ' Family with young childre...",720 day,51.507720,0.022981
1,Great Cumberland Place Westminster Borough Lon...,1190,8/3/2017,The Cumberland A Guoman Hotel,Gibraltar,No Negative,0,5180,The location was excellent rieally good next ...,11,2,"[' Leisure trip ', ' Group ', ' Standard Doubl...",0 days,51.514879,-0.160650


In [20]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103148 entries, 0 to 103147
Data columns (total 15 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               103148 non-null  object 
 1   Additional_Number_of_Scoring                103148 non-null  int64  
 2   Review_Date                                 103148 non-null  object 
 3   Hotel_Name                                  103148 non-null  object 
 4   Reviewer_Nationality                        103148 non-null  object 
 5   Negative_Review                             103148 non-null  object 
 6   Review_Total_Negative_Word_Counts           103148 non-null  int64  
 7   Total_Number_of_Reviews                     103148 non-null  int64  
 8   Positive_Review                             103148 non-null  object 
 9   Review_Total_Positive_Word_Counts           103148 non-null  int64  
 

In [21]:
# Проверка на пропуски в данных
display(train.isnull().sum())
test.isnull().sum()

Hotel_Address                                    0
Additional_Number_of_Scoring                     0
Review_Date                                      0
Hotel_Name                                       0
Reviewer_Nationality                             0
Negative_Review                                  0
Review_Total_Negative_Word_Counts                0
Total_Number_of_Reviews                          0
Positive_Review                                  0
Review_Total_Positive_Word_Counts                0
Total_Number_of_Reviews_Reviewer_Has_Given       0
Reviewer_Score                                   0
Tags                                             0
days_since_review                                0
lat                                           2600
lng                                           2600
dtype: int64

Hotel_Address                                   0
Additional_Number_of_Scoring                    0
Review_Date                                     0
Hotel_Name                                      0
Reviewer_Nationality                            0
Negative_Review                                 0
Review_Total_Negative_Word_Counts               0
Total_Number_of_Reviews                         0
Positive_Review                                 0
Review_Total_Positive_Word_Counts               0
Total_Number_of_Reviews_Reviewer_Has_Given      0
Tags                                            0
days_since_review                               0
lat                                           668
lng                                           668
dtype: int64

In [22]:
# Заполнение пропущенных координат
def fill_missing_coordinates(df):
    # Заполняем средними координатами по отелю
    hotel_coords = df.groupby('Hotel_Name')[['lat', 'lng']].transform('mean')
    df['lat'] = df['lat'].fillna(hotel_coords['lat'])
    df['lng'] = df['lng'].fillna(hotel_coords['lng'])
    
    # Если остались пропуски - заполняем общим средним
    df['lat'] = df['lat'].fillna(df['lat'].mean())
    df['lng'] = df['lng'].fillna(df['lng'].mean())
    return df

train = fill_missing_coordinates(train)
test = fill_missing_coordinates(test)

## Подготовка признаков

### Преобразование временных столбцов в числовой формат

In [23]:
# Преобразование столбца 'days_since_review' в числовой формат
train['days_since_review'] = train['days_since_review'].str.split(' ').str[0].astype(int)
test['days_since_review'] = test['days_since_review'].str.split(' ').str[0].astype(int)
train['days_since_review'].head(2)


0    108
1    434
Name: days_since_review, dtype: int64

In [24]:
# Преобразование столбца 'review_date' в datetime формат
# Преобразование и извлечение признаков — надёжно для train и test
for df in (train, test):
    # убедимся, что колонка есть и преобразуем в datetime
    if 'Review_Date' in df.columns:
        df['Review_Date'] = pd.to_datetime(df['Review_Date'], errors='coerce')
        df['review_year']  = df['Review_Date'].dt.year
        df['review_month'] = df['Review_Date'].dt.month
        df['review_day']   = df['Review_Date'].dt.day
        # день недели и сезон
        df['review_dayofweek'] = df['Review_Date'].dt.dayofweek
        df['review_season'] = df['review_month'] % 12 // 3 + 1  # 1: Winter, 2: Spring, 3: Summer, 4: Fall
    else:
        raise KeyError("В DataFrame нет колонки 'Review_Date'")

# Удаление исходного столбца 'Review_Date' после извлечения признаков
train.drop(columns=['Review_Date'], inplace=True)
test.drop(columns=['Review_Date'], inplace=True)

In [25]:
X_num_cols = ['Additional_Number_of_Scoring', 'Review_Total_Negative_Word_Counts',
              'Review_Total_Positive_Word_Counts', 'Total_Number_of_Reviews', 
              'Total_Number_of_Reviews_Reviewer_Has_Given', 'lat', 'lng', 
              'days_since_review', 'review_year', 'review_month', 'review_day', 'review_dayofweek', 'review_season', 
            ]

### Признаки тональности

In [26]:
# Папка и пути
sentiment_dir = "embeddings/sentiment_features"
os.makedirs(sentiment_dir, exist_ok=True)

sentiment_paths = {
    "train": f"{sentiment_dir}/train_sentiment.pkl",
    "test": f"{sentiment_dir}/test_sentiment.pkl",
    "feature_names": f"{sentiment_dir}/sentiment_feature_names.pkl"
}

def add_sentiment_features(df, save_path=None, batch_size=64):
    """
    Генерация признаков тональности с помощью cardiffnlp/twitter-roberta-base-sentiment-latest.
    Модель 3-классовая (POS/NEG/NEU), возвращает вероятности для каждого.
    Возвращает DataFrame с: pos, neg, neu, compound.
    """
    if save_path and os.path.exists(save_path):
        print(f"Загрузка сохранённых признаков тональности из: {save_path}")
        return pd.read_pickle(save_path)

    df = df.copy()
    # Объединяем отзывы, заменяя заглушки на пустые строки
    pos_reviews = df['Positive_Review'].replace("No Positive", "").fillna("")
    neg_reviews = df['Negative_Review'].replace("No Negative", "").fillna("")
    texts = (pos_reviews + " " + neg_reviews).str.strip().fillna("neutral")  # fallback на "neutral" для пустых

    # Инициализация pipeline
    device = 0 if torch.cuda.is_available() else -1
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
        device=device,
        batch_size=batch_size,
        truncation=True,
        max_length=512,
        return_all_scores=True  # Для получения всех вероятностей
    )

    print("Анализ тональности  через cardiffnlp/twitter-roberta-base-sentiment-latest...")

    # Разбиваем на batches для прогресса
    pos_scores, neg_scores, neu_scores = [], [], []
    num_batches = math.ceil(len(texts) / batch_size)
    for i in tqdm(range(num_batches), desc="Обработка batches"):
        batch_texts = texts[i*batch_size : (i+1)*batch_size]
        results = sentiment_pipeline(batch_texts.tolist())
        
        for res in results:
            scores = {r['label']: r['score'] for r in res}
            pos = scores.get('positive', 0.0)
            neg = scores.get('negative', 0.0)
            neu = scores.get('neutral', 0.0)
            pos_scores.append(pos)
            neg_scores.append(neg)
            neu_scores.append(neu)

    # Добавляем признаки
    df["pos"] = pos_scores
    df["neg"] = neg_scores
    df["neu"] = neu_scores
    df["compound"] = df["pos"] - df["neg"]  # Можно улучшить: (pos - neg) * (1 - neu) для учета neutral

    # Сохраняем
    if save_path:
        print(f"Сохранение признаков тональности в: {save_path}")
        df.to_pickle(save_path)

    return df

# --- Применение ---
print("Добавление признаков тональности ...")
train = add_sentiment_features(train, sentiment_paths["train"], batch_size=64)
test = add_sentiment_features(test, sentiment_paths["test"], batch_size=64)

# Сохраняем имена признаков
sentiment_cols = ['pos', 'neg', 'neu', 'compound']
with open(sentiment_paths["feature_names"], 'wb') as f:
    pickle.dump(sentiment_cols, f)

X_num_cols.extend(sentiment_cols)
print(f"Добавлено {len(sentiment_cols)} признаков тональности.")

Добавление признаков тональности ...
Загрузка сохранённых признаков тональности из: embeddings/sentiment_features/train_sentiment.pkl
Загрузка сохранённых признаков тональности из: embeddings/sentiment_features/test_sentiment.pkl
Добавлено 4 признаков тональности.


### Добавление target-based признаков

In [27]:
# === TARGET-BASED ПО HOTEL_NAME ===
alpha = 10
global_mean = train['Reviewer_Score'].mean()

# === 1. OOF для обучающей выборки ===
oof_vals = np.zeros(len(train))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, va_idx in kf.split(train):
    tr = train.iloc[tr_idx]
    va = train.iloc[va_idx]

    stats = tr.groupby('Hotel_Name')['Reviewer_Score'].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + alpha * global_mean) / (stats['count'] + alpha)

    oof_vals[va_idx] = va['Hotel_Name'].map(smooth).fillna(global_mean)

train['hotel_mean_score_smooth'] = oof_vals

# === 2. Полная статистика для теста ===
stats_full = train.groupby('Hotel_Name')['Reviewer_Score'].agg(['mean', 'count'])
smooth_full = (stats_full['mean'] * stats_full['count'] + alpha * global_mean) / (stats_full['count'] + alpha)
test['hotel_mean_score_smooth'] = test['Hotel_Name'].map(smooth_full).fillna(global_mean)

# === 3. Дополнительно: создаем hotel_count (опционально, но полезно) ===
train['hotel_count'] = train['Hotel_Name'].map(stats_full['count']).fillna(0)
test['hotel_count'] = test['Hotel_Name'].map(stats_full['count']).fillna(0)

# === 4. Теперь безопасно добавлять оба признака ===
X_num_cols.extend(['hotel_mean_score_smooth', 'hotel_count'])

print("Добавлены target-based признаки на основе Hotel_Name.")

# === TARGET-BASED ПО НАЦИОНАЛЬНОСТИ ===
alpha_nat = 5
oof_nat = np.zeros(len(train))
for tr_idx, va_idx in kf.split(train):
    tr = train.iloc[tr_idx]
    va = train.iloc[va_idx]
    stats_nat = tr.groupby('Reviewer_Nationality')['Reviewer_Score'].agg(['mean', 'count'])
    smooth_nat = (stats_nat['mean'] * stats_nat['count'] + alpha_nat * global_mean) / (stats_nat['count'] + alpha_nat)
    oof_nat[va_idx] = va['Reviewer_Nationality'].map(smooth_nat).fillna(global_mean)

train['nat_mean_score_smooth'] = oof_nat

stats_nat_full = train.groupby('Reviewer_Nationality')['Reviewer_Score'].agg(['mean', 'count'])
smooth_nat_full = (stats_nat_full['mean'] * stats_nat_full['count'] + alpha_nat * global_mean) / (stats_nat_full['count'] + alpha_nat)
test['nat_mean_score_smooth'] = test['Reviewer_Nationality'].map(smooth_nat_full).fillna(global_mean)

train['nat_count'] = train['Reviewer_Nationality'].map(stats_nat_full['count']).fillna(0)
test['nat_count'] = test['Reviewer_Nationality'].map(stats_nat_full['count']).fillna(0)

X_num_cols.extend(['nat_mean_score_smooth', 'nat_count'])
print("Добавлены target-based признаки по национальности")

Добавлены target-based признаки на основе Hotel_Name.
Добавлены target-based признаки по национальности


### Генерация текстовых эмбеддингов

#### Для отзывов

In [ ]:
def fine_tune_bge_model_regression(
    train_data,
    model_name='BAAI/bge-large-en-v1.5',
    output_path='fine_tuned_bge_regression',
    max_samples=200000,  
    num_pairs=300000,    
    epochs=2,
    batch_size=8
):
    """
    Supervised fine-tuning модели BGE для задачи регрессии оценки отеля.
    Эффективная генерация пар с масштабированием на большие данные.
    """

    print("Загрузка модели для regression fine-tuning...")
    model = SentenceTransformer(model_name, device='cuda')
    
    # Подвыборка для эффективности - УВЕЛИЧЕНО до 200K
    df = train_data.sample(n=min(max_samples, len(train_data)), random_state=42).copy()
    
    print(f"Используется {len(df)} примеров из {len(train_data)} доступных")
    
    # Подготовка текстов - объединяем оба отзыва
    df['combined_review'] = (
        "Hotel Review. Positive comments: " + df['Positive_Review'].replace("No Positive", "").fillna('') +
        " Negative comments: " + df['Negative_Review'].replace("No Negative", "").fillna('')
    )
    
    # ФИКСИРОВАННЫЙ диапазон оценок [2.5, 10.0]
    SCORE_MIN, SCORE_MAX = 2.5, 10.0
    df['norm_score'] = (df['Reviewer_Score'] - SCORE_MIN) / (SCORE_MAX - SCORE_MIN)
    
    print(f"Фиксированный диапазон оценок: {SCORE_MIN}-{SCORE_MAX}")
    print(f"Фактический диапазон в данных: {df['Reviewer_Score'].min():.1f}-{df['Reviewer_Score'].max():.1f}")
    
    # ЭФФЕКТИВНАЯ генерация пар - О(n) вместо O(n²)
    print(f"Генерация {num_pairs} пар для обучения...")
    
    train_examples = []
    
    # Стратегия 1: Пары с похожими оценками (разница < 1.0)
    print("Генерация пар с похожими оценками...")
    
    # Группируем по биннам оценок для быстрого поиска похожих
    df['score_bin'] = (df['Reviewer_Score'] * 2).astype(int)  # Бинны по 0.5
    bin_groups = df.groupby('score_bin')
    
    similar_pairs_needed = num_pairs // 2
    similar_pairs_created = 0
    
    for bin_val, group in tqdm(bin_groups, desc="Похожие пары по биннам"):
        if len(group) < 2:
            continue
            
        group_indices = group.index.tolist()
        np.random.shuffle(group_indices)
        
        # Создаем пары внутри группы
        for i in range(0, len(group_indices)-1, 2):
            if similar_pairs_created >= similar_pairs_needed:
                break
                
            idx1, idx2 = group_indices[i], group_indices[i+1]
            row1, row2 = df.loc[idx1], df.loc[idx2]
            
            # Вычисляем схожесть на основе разницы оценок
            score_diff = abs(row1['Reviewer_Score'] - row2['Reviewer_Score'])
            similarity_label = max(0.7, 1.0 - score_diff)  # Высокая схожесть для близких оценок
            
            train_examples.append(InputExample(
                texts=[row1['combined_review'], row2['combined_review']],
                label=float(similarity_label)
            ))
            similar_pairs_created += 1
    
    print(f"Создано {similar_pairs_created} пар с похожими оценками")
    
    # Стратегия 2: Случайные пары с метками на основе разницы оценок
    print("Генерация случайных пар...")
    
    random_pairs_needed = num_pairs - len(train_examples)
    indices = df.index.tolist()
    
    for i in tqdm(range(random_pairs_needed), desc="Случайные пары"):
        idx1, idx2 = np.random.choice(indices, 2, replace=False)
        row1, row2 = df.loc[idx1], df.loc[idx2]
        
        score_diff = abs(row1['Reviewer_Score'] - row2['Reviewer_Score'])
        
        # Нелинейная функция схожести
        if score_diff < 1.0:
            similarity_label = 0.8 + (1.0 - score_diff) * 0.2  # 0.8-1.0
        elif score_diff < 3.0:
            similarity_label = 0.5 + (3.0 - score_diff) * 0.15  # 0.5-0.8
        else:
            similarity_label = 0.1 + (5.0 - min(score_diff, 5.0)) * 0.1  # 0.1-0.5
        
        train_examples.append(InputExample(
            texts=[row1['combined_review'], row2['combined_review']],
            label=float(similarity_label)
        ))
    
    # Стратегия 3: Пары с самими собой для стабильности
    print("Добавление идентичных пар...")
    
    self_pairs_needed = min(10000, len(df))
    self_indices = np.random.choice(df.index.tolist(), self_pairs_needed, replace=False)
    
    for idx in tqdm(self_indices, desc="Идентичные пары"):
        row = df.loc[idx]
        train_examples.append(InputExample(
            texts=[row['combined_review'], row['combined_review']],
            label=1.0
        ))
    
    # Перемешиваем все примеры
    np.random.shuffle(train_examples)
    
    print(f"Итоговое количество пар: {len(train_examples)}")
    
    # DataLoader
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    
    # Используем CosineSimilarityLoss для регрессии
    train_loss = losses.CosineSimilarityLoss(model=model)
    
    # Fine-tuning
    print("Начинаем regression fine-tuning...")
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=epochs,
        warmup_steps=int(0.1 * len(train_dataloader)),
        show_progress_bar=True,
        optimizer_params={'lr': 1e-5},
        use_amp=True,
        checkpoint_path=os.path.join(output_path, 'checkpoint')
    )
    
    # Сохраняем модель
    os.makedirs(output_path, exist_ok=True)
    model.save(output_path)
    print(f"Regression fine-tuned модель сохранена в {output_path}")
    
    # Тестируем модель
    print("\nТестирование fine-tuned модели...")
    test_texts = df['combined_review'].head(100).tolist()
    test_embeddings = model.encode(test_texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    
    # Проверяем нормализацию
    norms = np.linalg.norm(test_embeddings, axis=1)
    print(f"Средняя норма эмбеддингов: {np.mean(norms):.3f} (должна быть ~1.0)")
    
    return model

# Запускаем улучшенный supervised fine-tuning
print("=== Улучшенный Regression Fine-tuning BGE модели ===")
tuned_model = fine_tune_bge_model_regression(
    train, 
    model_name='BAAI/bge-large-en-v1.5',
    max_samples=200000,    
    num_pairs=300000,      
    epochs=1,
    batch_size=8
)

print("Regression fine-tuning завершен успешно!")

In [29]:
def generate_embeddings_with_finetuned_model(
    train, 
    test, 
    use_fine_tuned=True,
    fine_tuned_model_path='fine_tuned_bge_regression'
):
    """
    Генерация эмбеддингов с использованием fine-tuned модели
    """
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import os
    
    emb_dir = "embeddings/reviews_finetuned/"
    os.makedirs(emb_dir, exist_ok=True)

    paths = {
        'train_neg': f"{emb_dir}/train_neg_emb.npy",
        'train_pos': f"{emb_dir}/train_pos_emb.npy", 
        'test_neg': f"{emb_dir}/test_neg_emb.npy", 
        'test_pos': f"{emb_dir}/test_pos_emb.npy",
        "train_pca": f"{emb_dir}/train_text_pca.npy",
        "test_pca": f"{emb_dir}/test_text_pca.npy"
    }

    # Проверяем существующие файлы
    pca_files_exist = all(os.path.exists(p) for p in [paths['train_pca'], paths['test_pca']])
    
    if pca_files_exist:
        print("PCA файлы найдены → загружаем...")
        X_text_train_full = np.load(paths["train_pca"])
        X_text_test_full = np.load(paths["test_pca"])
        return X_text_train_full, X_text_test_full

    # Загружаем модель
    if use_fine_tuned and os.path.exists(fine_tuned_model_path):
        print("Загрузка fine-tuned модели...")
        model = SentenceTransformer(fine_tuned_model_path, device='cuda')
        model_name = "fine-tuned BGE"
    else:
        print("Загрузка оригинальной BGE модели...")
        model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')
        model_name = "original BGE"
    
    print(f"Используется модель: {model_name}")

    def generate_embeddings_sentence_transformer(df, batch_size=32):
        """Генерация эмбеддингов через SentenceTransformer"""
        # Негативные отзывы с улучшенными промптами
        neg_texts = []
        for text in df['Negative_Review']:
            if text == "No Negative" or not text.strip():
                neg_texts.append("The guest reported no negative comments or complaints.")
            else:
                neg_texts.append(f"The guest reported negative aspects: {text}")
        
        # Позитивные отзывы с улучшенными промптами  
        pos_texts = []
        for text in df['Positive_Review']:
            if text == "No Positive" or not text.strip():
                pos_texts.append("The guest reported no positive comments or compliments.")
            else:
                pos_texts.append(f"The guest reported positive aspects: {text}")
        
        print("Генерация эмбеддингов для негативных отзывов...")
        neg_emb = model.encode(neg_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
        
        print("Генерация эмбеддингов для позитивных отзывов...")
        pos_emb = model.encode(pos_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
        
        return neg_emb, pos_emb

    # Проверяем существующие эмбеддинги
    emb_files_exist = all(os.path.exists(p) for p in [paths['train_neg'], paths['train_pos'], paths['test_neg'], paths['test_pos']])
    
    if not emb_files_exist:
        print("Генерация эмбеддингов с нуля...")
        
        print("Генерация для train данных...")
        train_neg_emb, train_pos_emb = generate_embeddings_sentence_transformer(train)
        
        print("Генерация для test данных...")
        test_neg_emb, test_pos_emb = generate_embeddings_sentence_transformer(test)
        
        # Сохраняем эмбеддинги
        np.save(paths['train_neg'], train_neg_emb)
        np.save(paths['train_pos'], train_pos_emb)
        np.save(paths['test_neg'], test_neg_emb)
        np.save(paths['test_pos'], test_pos_emb)
        print("Эмбеддинги сохранены!")
    else:
        print("Загрузка существующих эмбеддингов...")
        train_neg_emb = np.load(paths['train_neg'])
        train_pos_emb = np.load(paths['train_pos'])
        test_neg_emb = np.load(paths['test_neg'])
        test_pos_emb = np.load(paths['test_pos'])

    # Объединяем и применяем PCA
    print("Объединение эмбеддингов и применение PCA...")
    
    train_emb_combined = np.hstack([train_neg_emb, train_pos_emb])
    test_emb_combined = np.hstack([test_neg_emb, test_pos_emb])
    
    print(f"Объединенные эмбеддинги: {train_emb_combined.shape} (train), {test_emb_combined.shape} (test)")
    
    # PCA для уменьшения размерности
    from sklearn.decomposition import PCA
    
    pca = PCA()
    pca.fit(train_emb_combined)
    
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    n_comp = np.argmax(cum_var >= 0.95) + 1
    print(f"Выбрано {n_comp} компонент для 95% дисперсии")
    
    pca = PCA(n_components=n_comp)
    X_text_train_full = pca.fit_transform(train_emb_combined)
    X_text_test_full = pca.transform(test_emb_combined)
    
    # Сохраняем PCA результаты
    np.save(paths["train_pca"], X_text_train_full)
    np.save(paths["test_pca"], X_text_test_full)
    print("PCA результаты сохранены!")
    
    # Очистка памяти
    del model
    import gc
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Финальная размерность: {X_text_train_full.shape} (train), {X_text_test_full.shape} (test)")
    return X_text_train_full, X_text_test_full

# Запускаем генерацию эмбеддингов
print("=== Генерация эмбеддингов с Fine-tuned моделью ===")
use_finetuned = 'tuned_model' in locals() and tuned_model is not None

X_text_train_full, X_text_test_full = generate_embeddings_with_finetuned_model(
    train, 
    test, 
    use_fine_tuned=use_finetuned
)

=== Генерация эмбеддингов с Fine-tuned моделью ===
PCA файлы найдены → загружаем...


#### Для тэгов

In [31]:
# === ЭМБЕДДИНГИ ДЛЯ ТЭГОВ ===
gc.collect(); torch.cuda.empty_cache()

tag_emb_dir = "embeddings/tags"
os.makedirs(tag_emb_dir, exist_ok=True)

tag_paths = {
    "embeddings": f"{tag_emb_dir}/tag_embeddings.npy",
    "mapping": f"{tag_emb_dir}/tag_mapping.pkl"
}

# Функция парсинга тегов (если ещё не сделана)
def parse_tags(tags_str):
    tags_str = tags_str.strip("[]").replace("'", "").split(', ')
    return [tag.strip() for tag in tags_str if tag.strip()]

# Применяем парсинг, если parsed_tags ещё нет
if 'parsed_tags' not in train.columns:
    train['parsed_tags'] = train['Tags'].apply(parse_tags)
if 'parsed_tags' not in test.columns:
    test['parsed_tags'] = test['Tags'].apply(parse_tags)

# Получаем уникальные теги
all_tags = [tag for tags in pd.concat([train['parsed_tags'], test['parsed_tags']]) for tag in tags]
unique_tags = list(set(all_tags))
print(f"Уникальных тегов: {len(unique_tags)}")

if not all(os.path.exists(p) for p in tag_paths.values()):
    print("Генерация эмбеддингов тегов...")
    model = SentenceTransformer('BAAI/bge-m3', device='cuda')
    
    tag_texts = list(unique_tags)
    tag_emb = model.encode(tag_texts, batch_size=128, show_progress_bar=True,
                           normalize_embeddings=True, convert_to_numpy=True).astype('float16')
    
    # PCA до 95% дисперсии
    pca = PCA()
    pca.fit(tag_emb)
    
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    n_comp = np.argmax(cum_var >= 0.95) + 1
    print(f"Выбрано {n_comp} компонент для 95% дисперсии")
    
    pca = PCA(n_components=n_comp)
    tag_emb_pca = pca.fit_transform(tag_emb)
    
    tag_emb_dict = dict(zip(unique_tags, tag_emb_pca))
    
    np.save(tag_paths["embeddings"], tag_emb_pca)
    with open(tag_paths["mapping"], 'wb') as f:
        pickle.dump(tag_emb_dict, f)
    
    del model
    gc.collect(); torch.cuda.empty_cache()
else:
    print("Загрузка эмбеддингов тегов...")
    tag_emb_pca = np.load(tag_paths["embeddings"])
    with open(tag_paths["mapping"], 'rb') as f:
        tag_emb_dict = pickle.load(f)

# Функция для среднего эмбеддинга тегов в строке (поскольку тегов несколько)
def get_avg_tag_emb(tags, emb_dict, emb_dim=20):
    if not tags:
        return np.zeros(emb_dim, dtype=np.float32)
    embs = [emb_dict.get(tag, np.zeros(emb_dim)) for tag in tags]
    return np.mean(embs, axis=0)

train_tag_emb = np.array([get_avg_tag_emb(tags, tag_emb_dict) for tags in train['parsed_tags']])
test_tag_emb = np.array([get_avg_tag_emb(tags, tag_emb_dict) for tags in test['parsed_tags']])
print(f"Эмбеддинги тегов: {train_tag_emb.shape} (train), {test_tag_emb.shape} (test)")

Уникальных тегов: 2428
Загрузка эмбеддингов тегов...
Эмбеддинги тегов: (412590, 207) (train), (103148, 207) (test)


In [32]:
# === ОБЪЕДИНЕНИЕ ОТЕЛЕЙ, ОТЗЫВОВ И ТЭГОВ ===

X_text_train_full = np.hstack([X_text_train_full, train_tag_emb])
X_text_test_full = np.hstack([X_text_test_full, test_tag_emb])

print(f"Финальная размерность: {X_text_train_full.shape} (train), {X_text_test_full.shape} (test)")

Финальная размерность: (412590, 430) (train), (103148, 430) (test)


In [33]:

# Числовые + текстовые эмбеддинги → финальная матрица
X_full = np.hstack([
    train[X_num_cols].values,
    X_text_train_full
])

y_full = train['Reviewer_Score'].values

# Для теста (пока просто сохраняем — понадобится позже при сабмите)
X_test_full = np.hstack([
    test[X_num_cols].values,
    X_text_test_full
])

print(f"Готово! X_full: {X_full.shape}, y_full: {y_full.shape}")
print(f"X_test_full (для предсказания): {X_test_full.shape}")

Готово! X_full: (412590, 451), y_full: (412590,)
X_test_full (для предсказания): (103148, 451)


## Обучение модели

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.1, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# # Обучение на 30% данных
# X_train = X_train[:int(0.30 * len(X_train))]
# y_train = y_train[:int(0.30 * len(y_train))]

# print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (371331, 451), Test: (41259, 451)


In [35]:
# --- MAPE ---
def safe_mape(y_true, y_pred, eps=1e-6):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

# --- Optuna объективная функция (простое разделение train/validation) ---
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mape',
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 32, 800),
        'max_depth': trial.suggest_int('max_depth', -1, 40),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 300),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.3, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.3, 1.0),
        'bagging_freq': 1,
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 10.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 10.0),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'verbosity': -1,
        'seed': 42
    }
    
    # Разделение на train и validation (80/20)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    train_set = lgb.Dataset(X_tr, label=y_tr)
    valid_set = lgb.Dataset(X_va, label=y_va, reference=train_set)
    
    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    
    pred = model.predict(X_va)
    score = safe_mape(y_va, pred)
    
    return score

# --- Запуск Optuna  ---
print("Optuna (LightGBM + GPU)")
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)  # Уменьшаем warmup для ускорения
)
study.optimize(objective, n_trials=30)

print("Лучшие параметры:", study.best_params)
print("Лучший CV MAPE:   ", study.best_value)

# --- Финальная модель на train
print("Финальное обучение (train)...")

best_params = study.best_params.copy()
best_params.update({
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'verbosity': -1,
    'seed': 42
})

train_set = lgb.Dataset(X_train, label=y_train)
test_set  = lgb.Dataset(X_test, label=y_test, reference=train_set)

final_model = lgb.train(
    best_params,
    train_set,
    num_boost_round=3000,
    valid_sets=[test_set],
    callbacks=[
        lgb.early_stopping(150, first_metric_only=True, verbose=True),
        lgb.log_evaluation(100)
    ]
)

# --- Финальная оценка на локальном тесте ---
pred_test = final_model.predict(X_test)
final_mape = safe_mape(y_test, pred_test)

print(f"\nФинальный локальный MAPE: {final_mape:.4f}%")

[I 2025-11-18 01:54:21,454] A new study created in memory with name: no-name-4944129d-d844-4798-b29e-651513b9b69b


Optuna (LightGBM + GPU)


[I 2025-11-18 01:57:23,677] Trial 0 finished with value: 9.094824004538976 and parameters: {'learning_rate': 0.008468008575248327, 'num_leaves': 763, 'max_depth': 29, 'min_data_in_leaf': 188, 'feature_fraction': 0.40921304830970556, 'bagging_fraction': 0.40919616423534183, 'lambda_l1': 0.5808361216819946, 'lambda_l2': 8.661761457749352, 'min_gain_to_split': 0.6011150117432088}. Best is trial 0 with value: 9.094824004538976.
[I 2025-11-18 01:57:36,250] Trial 1 finished with value: 9.133422021112391 and parameters: {'learning_rate': 0.05675206026988748, 'num_leaves': 47, 'max_depth': 39, 'min_data_in_leaf': 253, 'feature_fraction': 0.44863737747479326, 'bagging_fraction': 0.42727747704497043, 'lambda_l1': 1.8340450985343382, 'lambda_l2': 3.0424224295953772, 'min_gain_to_split': 0.5247564316322378}. Best is trial 0 with value: 9.094824004538976.
[I 2025-11-18 01:59:58,112] Trial 2 finished with value: 9.080439376687163 and parameters: {'learning_rate': 0.01174843954800703, 'num_leaves': 2

Лучшие параметры: {'learning_rate': 0.011090784114084451, 'num_leaves': 311, 'max_depth': 23, 'min_data_in_leaf': 249, 'feature_fraction': 0.776520069136377, 'bagging_fraction': 0.8690430838992695, 'lambda_l1': 4.209696101823797, 'lambda_l2': 2.330191420184345, 'min_gain_to_split': 0.6131221261281926}
Лучший CV MAPE:    9.06061568481207
Финальное обучение (train)...
Training until validation scores don't improve for 150 rounds
[100]	valid_0's l2: 0.983666
[200]	valid_0's l2: 0.767516
[300]	valid_0's l2: 0.731055
[400]	valid_0's l2: 0.72135
[500]	valid_0's l2: 0.717296
[600]	valid_0's l2: 0.71513
[700]	valid_0's l2: 0.713755
[800]	valid_0's l2: 0.712826
[900]	valid_0's l2: 0.71219
[1000]	valid_0's l2: 0.711744
[1100]	valid_0's l2: 0.711341
[1200]	valid_0's l2: 0.711165
[1300]	valid_0's l2: 0.711003
[1400]	valid_0's l2: 0.710903
[1500]	valid_0's l2: 0.710762
[1600]	valid_0's l2: 0.710785
[1700]	valid_0's l2: 0.710693
[1800]	valid_0's l2: 0.710688
[1900]	valid_0's l2: 0.710544
[2000]	vali

In [36]:
# --- САБМИТ ---

# Предсказание на тестовых данных
predict_test = final_model.predict(X_test_full)

def postprocess_predictions(y_pred):
    y_pred = np.clip(y_pred, 2.5, 10.0)
    return np.round(y_pred, 1)  # Округляем до одного знака после запятой

predict_submit_clipped = postprocess_predictions(predict_test)  
print(f"Длина предсказаний: {len(predict_submit_clipped)}")

# Создание сабмита
res = pd.DataFrame()
res['id'] = range(len(predict_submit_clipped))
res['prediction'] = predict_submit_clipped

print("Первые 5 предсказаний:")
print(res.head())

# Сохранение в файл
res.to_csv('submission.csv', index=False)
print("Сабмит сохранен в submission.csv")

Длина предсказаний: 103148
Первые 5 предсказаний:
   id  prediction
0   0         9.3
1   1         9.1
2   2         7.8
3   3         9.9
4   4         8.5
Сабмит сохранен в submission.csv
